# 11 — Bằng chứng trước khi đưa ra lời khuyên

**Câu hỏi:** làm sao biết đang thiếu bài nào, thay vì chỉ nhìn hai số tổng?

Đọc theo thứ tự: đối chiếu train → xem lỗi nộp sớm → tạo hồ sơ demo. Không sửa CSV nghiên cứu hoặc model. Cần các CSV gốc và bảng cutoff; có thể đặt biến môi trường CLHT_RESEARCH_DATA nếu dữ liệu ở nơi khác.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display

current = Path.cwd()
ROOT = next(parent for parent in [current, *current.parents] if (parent / 'apps/backend/app').is_dir())
sys.path.insert(0, str(ROOT / 'apps'))
sys.path.insert(0, str(ROOT / 'apps/backend'))
RESEARCH = Path(os.environ.get('CLHT_RESEARCH_DATA', str(ROOT.parent / 'CLHT_GB_GITHUB')))
OUTPUT = ROOT / 'apps/research_outputs'
FIGURES = OUTPUT / 'hinh'
OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print('Mốc phục vụ app: 75%. Không đổi model và không dùng nhãn cuối kỳ để sinh gợi ý.')


Mốc phục vụ app: 75%. Không đổi model và không dùng nhãn cuối kỳ để sinh gợi ý.


## 1. Đối chiếu theo mã bài

Tập D gồm bài có hạn không sau cutoff; S gồm bài đã có bản ghi nộp đến cutoff. Cần đếm D trừ S. Nộp sớm một bài ngoài D không làm mất bài thiếu trong D. Bảo lưu được giữ như một bản ghi hoàn thành; ngoại lệ gia hạn vẫn cần xác nhận.

In [2]:
from research.prepare_cases import training_audit, build_cases
checks = []
for percent in (50, 75):
    audit, assessments, submissions, source = training_audit(RESEARCH, percent)
    checks.append({
        'cutoff': percent, 'dòng train': len(audit),
        'đếm thiếu thấp hơn đối chiếu': int((audit.missing_exact > audit.missing_submissions_to_t).sum()),
        'bài đến hạn chưa có bản ghi': int((audit.missing_exact > 0).sum()),
        'có nộp trễ': int((audit.late_unbanked_exact > 0).sum()),
    })
check_table = pd.DataFrame(checks)
display(check_table)
assert check_table['đếm thiếu thấp hơn đối chiếu'].tolist() == [406, 693]
check_table.to_csv(OUTPUT / 'doi_chieu_bai_nop.csv', index=False, encoding='utf-8-sig')

,cutoff,dòng train,đếm thiếu thấp hơn đối chiếu,bài đến hạn chưa có bản ghi,có nộp trễ
0,50,17310,406,5788,8199
1,75,16297,693,5783,8771


**Đọc kết quả:** 406 và 693 là số dòng train bị cách đếm cũ đánh giá thiếu thấp hơn phép đối chiếu mã bài, không phải số sinh viên chắc chắn bỏ học. Đây là kiểm tra dữ liệu, không phải đo hiệu quả lời khuyên. Nếu dữ liệu đổi, cần xem lại assertion thay vì sửa số cho chạy qua.

## 2. Ví dụ nhỏ có đáp án

Đến ngày 196: bài A hạn 100 đã nộp; B hạn 170 chưa nộp; C hạn 220 đã nộp sớm ngày 190. Tổng đến hạn = 2, tổng đã nộp = 2 nhưng vẫn thiếu bản ghi của B. Ví dụ này được tạo để kiểm thử, không phải một người học thật.

In [3]:
from app.schemas.support import LearningEvidence
from app.services.support import assessment_evidence
example = LearningEvidence(snapshot_day=196, assessments=[
    {'assessment_id': 'A', 'due_day': 100, 'submitted_day': 110},
    {'assessment_id': 'B', 'due_day': 170},
    {'assessment_id': 'C', 'due_day': 220, 'submitted_day': 190},
])
result = assessment_evidence(example)
display(pd.DataFrame([result]))
assert result['missing_ids'] == ['B']
assert result['early_future'] == 1

,due,submitted,completed_due,missing,late,missing_ids,late_ids,unknown_deadlines,exempt_due,early_future
0,2,2,1,1,1,[B],[A],0,0,1


## 3. Tạo hồ sơ minh họa cho app

Chọn dòng đầu tiên khớp mỗi tình huống trong train. Không chọn theo kết quả cuối kỳ hoặc theo xác suất model. Sáu bản trích rút thật và một tình huống thiếu xác nhận nguồn được tạo có chủ đích để kiểm thử. Không xuất mã sinh viên, nhãn hay điểm; mã bài đổi thành B01… Đây không phải bảo đảm ẩn danh tuyệt đối nếu đối chiếu lại dữ liệu gốc.

In [4]:
from app.schemas.support import SupportInput
from app.services.support import CASES_PATH, demo_cases
cases = build_cases(RESEARCH)
for case in cases:
    SupportInput.model_validate(case['payload'])
    assert not {'id_student', 'risk', 'final_result'} & case['payload']['features'].keys()
CASES_PATH.write_text(json.dumps(cases, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
demo_cases.cache_clear()
display(pd.DataFrame([{'mã tình huống': case['id'], 'mô tả': case['title'], 'nguồn': case['source_kind']} for case in cases]))
print('Đã tạo', len(cases), 'tình huống; chưa chạy dự báo trong notebook này.')

,mã tình huống,mô tả,nguồn
0,bai-den-han,Cần kiểm tra bài đến hạn,research_snapshot
1,nop-tre,"Đã nộp đủ, từng nộp trễ",research_snapshot
2,gian-doan-7,Một tuần chưa có tương tác,research_snapshot
3,gian-doan-30,Một tháng chưa có tương tác,research_snapshot
4,nop-som,Nộp sớm không bù được bài thiếu,research_snapshot
5,khong-khop,Chưa có tình huống cần gợi ý,research_snapshot
6,thieu-bang-chung,Thiếu bằng chứng: hệ thống từ chối,synthetic_test


Đã tạo 7 tình huống; chưa chạy dự báo trong notebook này.


## 4. Tự kiểm tra cách hiểu

Nếu đổi ngày nộp của C từ 190 thành 191, B có còn thiếu không? Có: C vẫn là bài có hạn sau cutoff. Hãy xem đáp án chạy ở cell tiếp theo.

In [5]:
changed = example.model_copy(deep=True)
changed.assessments[2].submitted_day = 191
assert assessment_evidence(changed)['missing_ids'] == ['B']
print('B vẫn cần kiểm tra. Không lấy bài nộp sớm để bù cho một mã bài khác.')

B vẫn cần kiểm tra. Không lấy bài nộp sớm để bù cho một mã bài khác.


## Giới hạn

Nguồn không ghi gia hạn cá nhân hoặc thời điểm công bố điểm. Bản đầu không gợi ý học lại chương cụ thể, không hứa tăng điểm và không đổi công thức 17 đặc trưng trong model cũ. Việc sinh gợi ý thực tế xem notebook 12.